# LipPY VALLR Fonem v1cc-FULL — Speed Perturbation + Label Smoothing + dim_ff=2048 + Dataset Mean/Std

**v1c sonuclari:** PER %49.1, WER %78.7 (Beam+KenLM). Bu notebook v1cc: 4 kritik iyilestirme.

## v1c → v1cc Degisiklikler

| Ozellik | v1c-full | v1cc-full | Neden |
|---|---|---|---|
| dim_ff | 1024 | **2048** | Standart Conformer orani (4x d_model) — temporal kapasite 2x |
| Speed perturbation | yok | **0.9x/1.0x/1.1x** | Efektif 3x veri artisi, konusma hizi varyasyonu |
| Label smoothing | yok | **0.1** | Viseme gruplari icin overconfidence azaltma |
| Mean/Std | 0.421/0.165 | **0.495/0.200** | Dataset gercek istatistigi (%17.5/%21 fark) |

## Beklenen sonuc

- v1c PER %49.1 → v1cc PER **%41-45** (toplam -4 ile -8 puan)
- WER (KenLM ile): %78.7 → **%70-74** tahmini

## Teknik detay

**Speed perturbation:** Video tensor temporal eksende F.interpolate ile 0.9x/1.0x/1.1x resize. Her sample icin rastgele hiz secilir.

**Label smoothing:** CTC loss native desteklemiyor. Cozum: `loss = (1-LS)*ctc + LS*uniform_kl`. Uniform KL = -log(1/V) sabit, efektif olarak confidence penalty.

**dim_ff=2048:** Model ~36M → ~50M param. T4 16GB'a sigar (BS=4, ACCUM=4).

**Dataset mean/std:** 500 ROI ornekleminden hesaplandi (609M piksel):
- MEAN = 0.495 (Auto-AVSR 0.421'den +17.5% fark)  
- STD = 0.200 (Auto-AVSR 0.165'ten +21.0% fark)

## Kaggle ayar

- `brokyewing/lippy-test` dataset input olarak ekle
- Auto-AVSR checkpoint: gdown otomatik
- 1 epoch ~55 dk (dim_ff buyudugu icin %15-20 yavaslar)
- 50 epoch ~46 saat → 2-3 Kaggle oturumu gerekir

**Hesap:** brokyewing  
**Checkpoint slug:** `brokyewing/lippy-phoneme-v1cc-full-checkpoint`

In [ ]:
!pip install -q torchaudio gdown

import os, glob, json, shutil, time, math, random, subprocess, re
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchaudio.models import Conformer
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import gdown

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'CPU')
print(f'numpy: {np.__version__}, torch: {torch.__version__}')

## 1) Config — FULL TRAINING

- `EPOCHS = 30`: Full training (Kaggle quota icin 30 ep, 35 yerine)
- `WARMUP_EPOCHS = 10`: Ep 1-9 augmentation kapali, ep 10+ aktif
- `ENABLE_COLOR_JITTER = True`: Brightness/contrast augmentation (speaker invariance)
- `ENABLE_TIMEMASK = True`: Temporal mask ep 10+
- `ENABLE_PIXEL_MASK = False`: Bizim mimaride patliyor (kanitlandi v1b++ tarihi)
- `ALPHA_ENT = 0.05`: Entropy regularization ep 10+
- `FORCE_SCRATCH = False`: Resume aktif (Kaggle 12h timeout sonrasi devam)

In [ ]:
# KULLANICI: Gerekirse path override et
CUSTOM_DATASET_BASE = None
PRETRAINED_CKPT = None      # Auto-discover: SADECE Auto-AVSR (v1b skip edilecek)

# Egitim hyperparametreleri — v1cc FULL TRAINING
EPOCHS = 50
LR_FE = 5e-6                # Pretrained frontend icin guvenli
LR_BE = 2e-4                # Scratch backend icin standart
WARMUP = 5                  # LR warmup (cosine schedule)
GCLIP = 2.0
ACCUM = 4
BS = 4
UPLOAD_EVERY = 5            # Her 5 epoch upload (Kaggle dataset version)

# ============ v1cc YENILIKLER ============
# 1) Label smoothing — viseme ambiguity icin overconfidence azaltma
LABEL_SMOOTHING = 0.1       # CTC loss'a uniform penalty ekler

# 2) Speed perturbation — temporal augmentation (efektif 3x veri)
ENABLE_SPEED_PERTURB = True
SPEED_FACTORS = [0.9, 1.0, 1.1]  # Yavas, normal, hizli

# 3) dim_ff artisi (model taniminda: DIM_FF = 2048)
# =============================================

# ============ AUGMENTATION (curriculum learning) ============
WARMUP_EPOCHS = 10          # Ep 1-9 augmentation KAPALI, ep 10+ aktif

# Color jitter — speaker invariance icin
ENABLE_COLOR_JITTER = True  # Ep 5+ aktif (curriculum'dan once kademeli)
COLOR_JITTER_FROM_EP = 5    # Hangi epoch'tan itibaren aktif
BRIGHTNESS_RANGE = 0.15     # +/- brightness shift
CONTRAST_RANGE = 0.15       # +/- contrast multiplier (1 +/- 0.15)

ENABLE_TIMEMASK = True      # Ep 10+ aktif
TM_WIN = 10
TM_STR = 25

ENABLE_PIXEL_MASK = False   # Bizim mimaride patliyor (kanitli)
PIXEL_MASK_PROB = 0.5
PM_RAMP_EPOCHS = 20

ALPHA_ENT = 0.05            # Ep 10+ aktif
# ============================================================

AVSR_URL = 'https://drive.google.com/uc?id=1shcWXUK2iauRhW9NbwCc25FjU1CoMm8i'

# Mutable state
class AugState:
    current_epoch = 1

def curr_pixel_mask_prob(epoch):
    if not ENABLE_PIXEL_MASK:
        return 0.0
    if epoch < WARMUP_EPOCHS:
        return 0.0
    ramp_pos = epoch - WARMUP_EPOCHS
    if ramp_pos >= PM_RAMP_EPOCHS:
        return PIXEL_MASK_PROB
    return PIXEL_MASK_PROB * (ramp_pos / PM_RAMP_EPOCHS)

print('Config (v1cc-full — speed perturb + label smooth + dim_ff=2048):')
print(f'  EPOCHS={EPOCHS}')
print(f'  LR_FE={LR_FE}, LR_BE={LR_BE}')
print(f'  LABEL_SMOOTHING={LABEL_SMOOTHING}')
print(f'  SPEED_PERTURB={ENABLE_SPEED_PERTURB} factors={SPEED_FACTORS}')
print(f'  DIM_FF=2048 (v1c: 1024)')
print(f'  MEAN/STD=0.495/0.200 (v1c: 0.421/0.165)')
print(f'  Augmentation curriculum:')
print(f'    Ep 1-{COLOR_JITTER_FROM_EP-1}: speed perturb only')
print(f'    Ep {COLOR_JITTER_FROM_EP}-{WARMUP_EPOCHS-1}: + color jitter')
print(f'    Ep {WARMUP_EPOCHS}+: + TimeMask + Entropy reg')
print(f'  Upload: her {UPLOAD_EVERY} epoch')
print(f'  Resume: aktif (Kaggle quota icin)')

## 2) Vocab — 31 Fonem (SPACE DAHIL, pilot F1=0.84 kanıtladı)

v1b'de SPACE token yoktu. Burada SPACE ayrı fonem: model kelime aralarını öğrenir.

In [ ]:
PHONEMES = [
    'SIL',     # 0: CTC blank
    'SPACE',   # 1: kelime ayiraci (YENI v1b++ — pilot F1=0.84 kanitladi)
    'a', 'e', 'i', 'o', 'u',
    '\u0131',  # i
    '\u00f6',  # o
    '\u00fc',  # u
    'b', 'p', 'm',
    'f', 'v',
    'd', 't', 'n', 'l',
    's', 'z',
    'c', '\u00e7',
    '\u015f', 'j',
    'k', 'g', '\u011f',
    'r', 'h', 'y',
]

BLANK_IDX = 0
SPACE_IDX = 1
PH_TO_IDX = {p: i for i, p in enumerate(PHONEMES)}
IDX_TO_PH = {i: p for i, p in enumerate(PHONEMES)}
NUM_PHONEMES = len(PHONEMES)
assert NUM_PHONEMES == 31, f'Beklenen 31, gelen {NUM_PHONEMES}'

CHAR_TO_PH = {ph: ph for ph in PHONEMES if ph not in ('SIL', 'SPACE')}

def text_to_phonemes(text):
    """SPACE dahil: text'teki bosluklari SPACE token'ina cevir."""
    phs = []
    for ch in text.lower().strip():
        if ch == ' ':
            # Ardisik bosluklari tek SPACE
            if not phs or phs[-1] != SPACE_IDX:
                phs.append(SPACE_IDX)
        elif ch in CHAR_TO_PH:
            phs.append(PH_TO_IDX[CHAR_TO_PH[ch]])
    # Bas/son SPACE temizle
    while phs and phs[0] == SPACE_IDX: phs.pop(0)
    while phs and phs[-1] == SPACE_IDX: phs.pop()
    return phs

def phonemes_to_text(indices):
    if isinstance(indices, torch.Tensor):
        indices = indices.tolist()
    out = []
    for i in indices:
        if i == BLANK_IDX:
            continue
        if i == SPACE_IDX:
            out.append(' ')
        else:
            out.append(IDX_TO_PH.get(i, ''))
    return ''.join(out)

test = 'merhaba d\u00fcnya nasilsin'
enc = text_to_phonemes(test)
dec = phonemes_to_text(enc)
print(f'Fonem: {NUM_PHONEMES} (SPACE DAHIL — pilot F1=0.84)')
print(f'Test: "{test}"')
print(f'  Encoded: {enc}')
print(f'  SPACE lokasyonlari: {[i for i, t in enumerate(enc) if t == SPACE_IDX]}')
print(f'  Decoded: "{dec}"')

## 3) Model (v1b ile AYNI mimari)

In [ ]:
class SiLU(nn.Module):
    def forward(self, x): return x * torch.sigmoid(x)

class BasicBlockSiLU(nn.Module):
    expansion = 1
    def __init__(self, ic, oc, stride=1, ds=None):
        super().__init__()
        self.conv1 = nn.Conv2d(ic, oc, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(oc)
        self.conv2 = nn.Conv2d(oc, oc, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(oc)
        self.act = SiLU()
        self.ds = ds
    def forward(self, x):
        identity = x if self.ds is None else self.ds(x)
        return self.act(self.bn2(self.conv2(self.act(self.bn1(self.conv1(x))))) + identity)

class ResNetSiLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = self._ml(64, 64, 2, 1)
        self.layer2 = self._ml(64, 128, 2, 2)
        self.layer3 = self._ml(128, 256, 2, 2)
        self.layer4 = self._ml(256, 512, 2, 2)
    def _ml(self, ic, oc, n, s):
        ds = nn.Sequential(nn.Conv2d(ic, oc, 1, stride=s, bias=False), nn.BatchNorm2d(oc)) if s != 1 or ic != oc else None
        return nn.Sequential(BasicBlockSiLU(ic, oc, s, ds), *[BasicBlockSiLU(oc, oc) for _ in range(n-1)])
    def forward(self, x): return self.layer4(self.layer3(self.layer2(self.layer1(x))))

D_MODEL = 512
ENC_LAYERS = 6
NHEAD = 8
DIM_FF = 2048       # << v1cc: 1024 -> 2048 (standart 4x d_model orani)
DROPOUT = 0.1

class LipPhonemeModel(nn.Module):
    def __init__(self, num_phonemes=NUM_PHONEMES, d_model=D_MODEL,
                 nhead=NHEAD, enc_layers=ENC_LAYERS, dim_ff=DIM_FF, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.frontend3D = nn.Sequential(
            nn.Conv3d(1, 64, (5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False),
            nn.BatchNorm3d(64), SiLU(),
            nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
        )
        self.trunk = ResNetSiLU()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(512, d_model) if d_model != 512 else nn.Identity()
        self.conformer = Conformer(
            input_dim=d_model, num_heads=nhead, ffn_dim=dim_ff,
            num_layers=enc_layers, depthwise_conv_kernel_size=31, dropout=dropout)
        self.ctc_fc = nn.Linear(d_model, num_phonemes)

    def forward(self, x, il=None):
        B, T, H, W = x.shape
        x = x.unsqueeze(1)
        x = self.frontend3D(x)
        B2, C, To, H2, W2 = x.shape
        x = x.permute(0, 2, 1, 3, 4).reshape(B2 * To, C, H2, W2)
        x = self.avg_pool(self.trunk(x)).view(B2, To, -1)
        x = self.proj(x)
        if il is None:
            il = torch.full((B2,), To, dtype=torch.long, device=x.device)
        x, _ = self.conformer(x, lengths=il)
        logits = self.ctc_fc(x).log_softmax(-1)
        return logits.permute(1, 0, 2)

total_p = sum(p.numel() for p in LipPhonemeModel().parameters())
print(f'Model: d={D_MODEL}, enc={ENC_LAYERS}, dim_ff={DIM_FF}, num_phonemes={NUM_PHONEMES}')
print(f'  Toplam param: {total_p:,} (v1c: ~36M, v1cc: ~50M)')
del total_p

## 4) Baslangic — Auto-AVSR Transfer (v1b SKIP)

**v1c transfer testi:**

1. **v1b ep75 SKIP** — `v1b_cands = []` zorla bos liste (test temizligi)
2. **Auto-AVSR frontend transfer** — vsr_trlrs3vox2_base.pth
   - frontend3D + trunk yuklenir (~11M params)
   - Conformer + ctc_fc scratch (~25M params)
3. **Scratch fallback** — Auto-AVSR de yoksa UYARI (manuel mudahale gerekir)

**Kritik duzeltme (v1b++ notebook'unda eksikti):**

```python
# 1. Prefix strip: 'frontend.frontend3D.' -> 'frontend3D.'
# 2. Downsample rename: '.downsample.' -> '.ds.'  (BIZIM mimarimiz 'ds' kullaniyor)
```

**Input olarak `brokyewing/lippy-test` (43K data) eklemeyi unutma!**
**Auto-AVSR checkpoint:** `gdown` ile otomatik indirilir, ya da Kaggle dataset'e ekle.

In [ ]:
model = LipPhonemeModel().to(DEVICE)

# ============ v1c: ZORLA Auto-AVSR Transfer (v1b SKIP) ============
transfer_done = False

# ASAMA 1: v1b ep75 ZORLA SKIP — v1c saf Auto-AVSR transfer testi
v1b_cands = []  # << v1c: bos liste, v1b transferi atlanacak
print('v1c: v1b ep75 transferi ZORLA atlandi (test temizligi)')

# ASAMA 2: Auto-AVSR frontend transfer — DOGRU mapping ile
print('Auto-AVSR transfer baslatiliyor...')
AVSR_CKPT = '/kaggle/working/vsr_trlrs3vox2_base.pth'
cands = glob.glob('/kaggle/input/**/vsr_trlrs3vox2_base.pth', recursive=True)
if cands:
    AVSR_CKPT = cands[0]
    print(f'Auto-AVSR Kaggle dataset\'te bulundu: {AVSR_CKPT}')
elif not os.path.exists(AVSR_CKPT):
    print('Auto-AVSR Google Drive\'dan indiriliyor (~1 GB)...')
    gdown.download(AVSR_URL, AVSR_CKPT, quiet=False)

if os.path.exists(AVSR_CKPT):
    ck = torch.load(AVSR_CKPT, map_location='cpu', weights_only=False)
    state = ck.get('model_state_dict', ck.get('state_dict', ck))
    print(f'Auto-AVSR checkpoint yuklendi, key sayisi: {len(state)}')

    # Sample keys (debug)
    print('Ilk 5 source key:')
    for k in list(state.keys())[:5]:
        print(f'  {k}: {list(state[k].shape)}')

    # ============ KRITIK MAPPING (v1c duzeltmesi) ============
    # Kaynak (Auto-AVSR E2E):    frontend.frontend3D.0.weight
    #                            frontend.trunk.layer1.0.conv1.weight
    #                            frontend.trunk.layer1.0.downsample.0.weight  <- bizim 'ds'
    # Hedef (LipPhonemeModel):   frontend3D.0.weight
    #                            trunk.layer1.0.conv1.weight
    #                            trunk.layer1.0.ds.0.weight                   <- 'ds' adi
    fw = {}
    for k, v in state.items():
        # Sadece visual encoder (frontend.frontend3D + frontend.trunk) ilgileniyor
        if not (k.startswith('frontend.frontend3D.') or k.startswith('frontend.trunk.')):
            continue
        # 1. Prefix strip: 'frontend.' -> ''
        nk = k[len('frontend.'):]
        # 2. Downsample rename: '.downsample.' -> '.ds.'
        nk = nk.replace('.downsample.', '.ds.')
        fw[nk] = v

    # Bizim modele yukle
    ms = model.state_dict()
    matched, mismatched = [], []
    for k, v in fw.items():
        if k in ms:
            if ms[k].shape == v.shape:
                ms[k] = v
                matched.append(k)
            else:
                mismatched.append(f'{k}: src{list(v.shape)} vs tgt{list(ms[k].shape)}')

    model.load_state_dict(ms, strict=False)
    print(f'\nAuto-AVSR transfer SONUC:')
    print(f'  Matched: {len(matched)} agirlik yuklendi')
    print(f'  Mismatched: {len(mismatched)} (skip)')

    if mismatched:
        print(f'  Mismatch ornekleri:')
        for m in mismatched[:3]:
            print(f'    - {m}')

    # Verify: kac tane downsample/ds yuklendi (kritik)
    ds_loaded = sum(1 for k in matched if '.ds.' in k)
    print(f'  Downsample (.ds.) agirliklari: {ds_loaded} (beklenen: 6 = layer2-4 her birinde 2 katman)')

    # Kategori bazinda
    fe3d_loaded = sum(1 for k in matched if k.startswith('frontend3D.'))
    trunk_loaded = sum(1 for k in matched if k.startswith('trunk.'))
    print(f'  frontend3D: {fe3d_loaded} agirlik')
    print(f'  trunk:      {trunk_loaded} agirlik')

    transfer_done = (len(matched) > 0)

# ASAMA 3: Hicbir sey yoksa scratch
if not transfer_done:
    print('\nUYARI: Auto-AVSR yuklenemedi — TAMAMEN SCRATCH!')
    print('  Kaggle dataset\'e vsr_trlrs3vox2_base.pth ekle veya gdown\'u kontrol et.')

total = sum(p.numel() for p in model.parameters())
print(f'\nModel hazir: {total:,} parametre (Auto-AVSR transfer={transfer_done})')

## 5) Veri + Curriculum Augmentation (3 Asama)

**Asama 1 — Ep 1-4: Saf veri** (color jitter dahi yok)
- Sadece geometric: random crop + flip
- Model temel feature ogrenir

**Asama 2 — Ep 5-9: Color jitter eklendi**
- Brightness ±0.15, Contrast ±15%
- Speaker invariance icin (lighting/cilt tonu varyasyonu)
- TimeMask hala kapali (cok early)

**Asama 3 — Ep 10+: Full augmentation**
- TimeMask (temporal occlusion)
- Color jitter (devam ediyor)
- Entropy regularization (alpha=0.05)

`augment(roi, train)` AugState.current_epoch'u okur, akademik curriculum learning standart pratiği.

In [ ]:
if CUSTOM_DATASET_BASE and os.path.exists(CUSTOM_DATASET_BASE):
    BASE = CUSTOM_DATASET_BASE
    ROIS_DIR = os.path.join(BASE, 'rois_mp4')
    TEXT_DIR = os.path.join(BASE, 'texts')
    SPLIT_DIR = os.path.join(BASE, 'splits')
    print(f'Custom dataset: {BASE}')
else:
    rois_cands = glob.glob('/kaggle/input/**/rois_mp4', recursive=True)
    ROIS_DIR = rois_cands[0] if rois_cands else '/kaggle/input/lippy-test/dataset/rois_mp4'
    BASE = os.path.dirname(ROIS_DIR)
    TEXT_DIR = os.path.join(BASE, 'texts')

    if os.path.exists(os.path.join(os.path.dirname(BASE), 'splits', 'train.txt')):
        SPLIT_DIR = os.path.join(os.path.dirname(BASE), 'splits')
    elif os.path.exists(os.path.join(BASE, 'splits', 'train.txt')):
        SPLIT_DIR = os.path.join(BASE, 'splits')
    else:
        SPLIT_DIR = os.path.join(BASE, 'splits')

    print(f'Auto-discovered: {BASE}')
    print(f'Split dir: {SPLIT_DIR}')

tp = os.path.join(SPLIT_DIR, 'train.txt')
vp = os.path.join(SPLIT_DIR, 'val.txt')
if not (os.path.exists(tp) and os.path.getsize(tp) > 0):
    SPLIT_DIR = '/kaggle/working/splits'
    os.makedirs(SPLIT_DIR, exist_ok=True)
    tp = os.path.join(SPLIT_DIR, 'train.txt')
    vp = os.path.join(SPLIT_DIR, 'val.txt')
    rids = sorted([f[:-4] for f in os.listdir(ROIS_DIR) if f.endswith('.mp4')])
    tids = set(f[:-4] for f in os.listdir(TEXT_DIR) if f.endswith('.txt'))
    valid = [s for s in rids if s in tids]
    random.seed(42)
    random.shuffle(valid)
    si = int(len(valid) * 0.85)
    open(tp, 'w').write('\n'.join(sorted(valid[:si])) + '\n')
    open(vp, 'w').write('\n'.join(sorted(valid[si:])) + '\n')
    print(f'Splits: train={si} val={len(valid) - si}')

# v1cc: Dataset-specific normalization (500 ROI ornekleminden hesaplandi)
MEAN, STD, CROP = 0.495, 0.200, 88
print(f'  Normalization: MEAN={MEAN}, STD={STD} (dataset-specific, NOT Auto-AVSR)')

def read_mp4(p):
    cap = cv2.VideoCapture(p)
    frames = []
    while True:
        r, fr = cap.read()
        if not r: break
        frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY))
    cap.release()
    return np.array(frames, dtype=np.uint8) if frames else None

def speed_perturb(roi):
    """Temporal speed perturbation: 0.9x, 1.0x veya 1.1x hiz.
    roi: np.ndarray (T, H, W) float32.
    Returns: np.ndarray (T_new, H, W) float32.
    """
    factor = random.choice(SPEED_FACTORS)
    if factor == 1.0:
        return roi
    T, H, W = roi.shape
    new_T = max(5, int(round(T / factor)))
    roi_tensor = torch.from_numpy(roi).unsqueeze(0).unsqueeze(0)  # (1, 1, T, H, W)
    roi_resized = F.interpolate(roi_tensor, size=(new_T, H, W), mode='trilinear', align_corners=False)
    return roi_resized.squeeze(0).squeeze(0).numpy()

def time_mask(roi):
    T = roi.shape[0]
    for _ in range(math.ceil(T / TM_STR)):
        ml = random.randint(0, TM_WIN - 1)
        if ml > 0:
            ms = random.randint(0, max(0, T - ml))
            roi[ms:ms + ml] = 0.0
    return roi

def pixel_mask(roi, p):
    T, H, W = roi.shape
    mask = (np.random.rand(T, H, W) > p).astype(np.float32)
    return roi * mask

def color_jitter(roi):
    if random.random() < 0.5:
        brightness_shift = random.uniform(-BRIGHTNESS_RANGE, BRIGHTNESS_RANGE)
        roi = roi + brightness_shift
    if random.random() < 0.5:
        contrast_scale = 1.0 + random.uniform(-CONTRAST_RANGE, CONTRAST_RANGE)
        roi = (roi - 0.5) * contrast_scale + 0.5
    roi = np.clip(roi, 0.0, 1.0)
    return roi

def augment(roi, train):
    roi = roi.astype(np.float32) / 255.0
    T, H, W = roi.shape

    # v1cc: Speed perturbation (her zaman aktif, curriculum'dan bagimsiz)
    if train and ENABLE_SPEED_PERTURB:
        roi = speed_perturb(roi)
        T = roi.shape[0]  # T degismis olabilir

    if train:
        t = random.randint(0, max(0, H - CROP))
        l = random.randint(0, max(0, W - CROP))
        if random.random() > 0.5:
            roi = roi[:, :, ::-1].copy()
    else:
        t = (H - CROP) // 2
        l = (W - CROP) // 2
    roi = roi[:, t:t + CROP, l:l + CROP]

    # ======= CURRICULUM (3 asama) =======
    if train:
        ep = AugState.current_epoch

        # ASAMA 2+ (ep >= COLOR_JITTER_FROM_EP): color jitter
        if ENABLE_COLOR_JITTER and ep >= COLOR_JITTER_FROM_EP:
            roi = color_jitter(roi)

        # ASAMA 3 (ep >= WARMUP_EPOCHS): TimeMask + Entropy
        if ep >= WARMUP_EPOCHS:
            if ENABLE_TIMEMASK:
                roi = time_mask(roi)
            if ENABLE_PIXEL_MASK:
                pm_p = curr_pixel_mask_prob(ep)
                if pm_p > 0:
                    roi = pixel_mask(roi, p=pm_p)
    # ======================================

    return torch.from_numpy(((roi - MEAN) / STD).copy()).float()

class DS(Dataset):
    def __init__(self, split):
        self.train = split == 'train'
        with open(os.path.join(SPLIT_DIR, f'{split}.txt'), 'r', encoding='utf-8') as f:
            aids = [l.strip() for l in f if l.strip()]
        self.ids = []
        sk = 0
        for s in aids:
            rp = os.path.join(ROIS_DIR, f'{s}.mp4')
            tp_ = os.path.join(TEXT_DIR, f'{s}.txt')
            if not os.path.exists(rp) or not os.path.exists(tp_):
                sk += 1; continue
            cap = cv2.VideoCapture(rp)
            nf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            if nf < 5:
                sk += 1; continue
            with open(tp_, 'r', encoding='utf-8') as f:
                txt = f.read().strip()
            nc = len(text_to_phonemes(txt))
            if nc == 0 or nf / nc < 1.2:
                sk += 1; continue
            self.ids.append(s)
        print(f'  [{split}] {len(self.ids)} ornek ({sk} atildi)')

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        for o in range(len(self.ids)):
            s = self.ids[(i + o) % len(self.ids)]
            try:
                roi = read_mp4(os.path.join(ROIS_DIR, f'{s}.mp4'))
                if roi is None or len(roi) < 5:
                    continue
                v = augment(roi, self.train)
                with open(os.path.join(TEXT_DIR, f'{s}.txt'), 'r', encoding='utf-8') as f:
                    txt = f.read().strip()
                lab = torch.tensor(text_to_phonemes(txt), dtype=torch.long)
                if len(lab) == 0:
                    continue
                return v, lab
            except Exception:
                continue
        raise RuntimeError('ornek yuklenemedi')

def collate(batch):
    vs, ls = zip(*batch)
    il = torch.tensor([v.shape[0] for v in vs], dtype=torch.long)
    pad = torch.zeros(len(vs), il.max().item(), CROP, CROP)
    for i, v in enumerate(vs):
        pad[i, :v.shape[0]] = v
    tl = torch.tensor([l.shape[0] for l in ls], dtype=torch.long)
    return pad, torch.cat(ls), il, tl

trl = DataLoader(DS('train'), batch_size=BS, shuffle=True, collate_fn=collate, num_workers=0, pin_memory=True)
vll = DataLoader(DS('val'), batch_size=BS, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)
print(f'Train: {len(trl)} batch | Val: {len(vll)} batch')

## 5b) Speaker overlap check — kritik!

Kullanıcı gözlemi: v1b++ ep35 görmediği kişilerde kötü sonuç veriyor (speaker dependence). Bu cell train/val split'inde **speaker leak** olup olmadığını kontrol eder.

**Tipik dosya formatı:**
- `gokcenin_0042.mp4` → speaker = "gokcenin"
- Random shuffle ile split aynı kanal/kişiyi hem train hem val'a koyabilir

**Eğer overlap %80+ ise:** Mevcut val skoru gerçek generalization'ı yansıtmıyor — sonuç şişmiş olabilir. v1c-full training'de speaker-disjoint split uygulanmalı.

In [ ]:
# Speaker overlap check — sample ID'lerden speaker prefix tespit
from collections import Counter

def detect_speaker(sample_id):
    """Sample ID'den speaker tespit. Yaygin formatlar:
       'gokcenin_0042' -> 'gokcenin'
       'speaker01_seg42' -> 'speaker01'
       'abc12345' -> None (hash, prefix yok)
    """
    parts = sample_id.split('_')
    if len(parts) >= 2 and not parts[0].isdigit():
        # Ilk parca alphanum, ikinci ise sayi → muhtemelen speaker_segment
        return parts[0]
    return None

# Train ve val ID'lerini oku
with open(tp, 'r', encoding='utf-8') as f:
    train_ids = [l.strip() for l in f if l.strip()]
with open(vp, 'r', encoding='utf-8') as f:
    val_ids = [l.strip() for l in f if l.strip()]

print(f'Train: {len(train_ids)} | Val: {len(val_ids)}')

# Speaker tespit
train_speakers = [detect_speaker(s) for s in train_ids]
val_speakers = [detect_speaker(s) for s in val_ids]

train_speaker_set = set(s for s in train_speakers if s)
val_speaker_set = set(s for s in val_speakers if s)
none_count_train = sum(1 for s in train_speakers if s is None)
none_count_val = sum(1 for s in val_speakers if s is None)

print(f'\nSample ID format analizi:')
print(f'  Speaker prefix tespit edilen train: {len(train_speakers) - none_count_train}/{len(train_ids)}')
print(f'  Speaker prefix tespit edilen val:   {len(val_speakers) - none_count_val}/{len(val_ids)}')
if none_count_train > 0:
    print(f'  Tespit edilemeyen ornekler: {none_count_train} train, {none_count_val} val')
    print(f'  Ornek ID\'ler:')
    for s in train_ids[:5]:
        print(f'    - {s} -> speaker: {detect_speaker(s)}')

# Speaker overlap
if train_speaker_set and val_speaker_set:
    overlap = train_speaker_set & val_speaker_set
    only_train = train_speaker_set - val_speaker_set
    only_val = val_speaker_set - train_speaker_set

    print(f'\nSpeaker analizi:')
    print(f'  Train speakers: {len(train_speaker_set)}')
    print(f'  Val speakers:   {len(val_speaker_set)}')
    print(f'  OVERLAP:        {len(overlap)} ({len(overlap)*100/max(len(val_speaker_set),1):.0f}% val\'in train\'de var)')
    print(f'  Sadece train:   {len(only_train)}')
    print(f'  Sadece val:     {len(only_val)}')

    if len(overlap) >= len(val_speaker_set) * 0.8:
        print(f'\n  >>> UYARI: Speaker leak var! Val\'in {len(overlap)*100//max(len(val_speaker_set),1)}% train ile ayni speaker.')
        print(f'      Bu val skoru gercek generalization\'i olcmuyor.')
        print(f'      v1c-full\'da speaker-disjoint split ON ERILMELI.')
    elif len(overlap) >= len(val_speaker_set) * 0.5:
        print(f'\n  >>> KIYI: Yari yariya speaker overlap. Val skoru kismi-shisirilmis.')
    else:
        print(f'\n  >>> OK: Speaker overlap dusuk, val skoru guvenilir.')

    # En cok sample veren ilk 5 speaker
    train_counter = Counter(train_speakers)
    if None in train_counter:
        del train_counter[None]
    top_train = train_counter.most_common(5)
    print(f'\nEn cok sample veren 5 speaker (train):')
    for sp, cnt in top_train:
        print(f'  {sp}: {cnt} ornek')
else:
    print(f'\n  >>> UYARI: Speaker prefix tespit edilemedi.')
    print(f'      Sample ID formati farkli (hash gibi). Manuel speaker tagging gerekir.')
    print(f'      Bu durumda v1c sonucu olcumu icin SPEAKER LEAK BELIRSIZ.')

## 6) Egitim — 30 Epoch (Full Training)

**Curriculum:**
- Ep 1-4: Saf veri (sadece geometric)
- Ep 5-9: + Color jitter (speaker invariance baslangic)
- Ep 10+: + TimeMask + Entropy reg (full augmentation)

**Resume aktif** (`FORCE_SCRATCH=False`):
- Kaggle 12h timeout'tan sonra notebook tekrar baslatildiginda kaldigi yerden devam
- best_model.pt PER-all bazinda kaydedilir
- checkpoint.pt her epoch sonu (resume icin)
- Upload her 5 epoch (Kaggle dataset version)

**Hedefe ulasinca durdurma yok** — 30 epoch sonuna kadar git. En iyi checkpoint best_model.pt'de saklanir.

In [ ]:
MDIR = '/kaggle/working/ckpt_phoneme_v1cc_full'
UDIR = '/kaggle/working/upload_phoneme_v1cc_full'
os.makedirs(MDIR, exist_ok=True)
os.makedirs(UDIR, exist_ok=True)
CP = f'{MDIR}/checkpoint.pt'
BP = f'{MDIR}/best_model.pt'

# ============ KAGGLE CREDENTIALS ============
KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', 'UNKNOWN')
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    KAGGLE_USER = _us.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
    os.environ['KAGGLE_KEY'] = _us.get_secret('KAGGLE_KEY')
except Exception as e:
    print(f'UYARI: Secrets okunamadi: {e}')
print(f'Kaggle user (upload kimligi): {KAGGLE_USER}')

DATASET_SLUG = 'lippy-phoneme-v1cc-full-checkpoint'
DATASET_ID = f'{KAGGLE_USER}/{DATASET_SLUG}'
with open(f'{UDIR}/dataset-metadata.json', 'w') as f:
    json.dump({'title': 'LipPY Phoneme v1cc-Full (Speed Perturb + Label Smooth + dim_ff=2048)',
               'id': DATASET_ID,
               'licenses': [{'name': 'CC0-1.0'}]}, f)
print(f'Upload target: {DATASET_ID}')

# ============ RESUME SAFETY ============
FORCE_SCRATCH = False       # v1cc: Resume aktif (ikinci oturum icin)
MAX_RESUME_PER = 0.95

def greedy_ctc(lp):
    ids = lp.argmax(-1).tolist()
    d = []
    p = -1
    for i in ids:
        if i != p and i != BLANK_IDX:
            d.append(i)
        p = i
    return d

def edit_dist(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for ai in a:
        r = [dp[0] + 1]
        for j, bj in enumerate(b):
            r.append(min(r[-1] + 1, dp[j + 1] + 1, dp[j] + (0 if ai == bj else 1)))
        dp = r
    return dp[-1]

def per_calc(preds, refs):
    td = tl = 0
    for p, r in zip(preds, refs):
        td += edit_dist(p, r)
        tl += max(len(r), 1)
    return td / max(tl, 1)

def v1c_metrics(preds, refs):
    per_all = per_calc(preds, refs)
    preds_ph = [[t for t in p if t != SPACE_IDX] for p in preds]
    refs_ph = [[t for t in r if t != SPACE_IDX] for r in refs]
    per_phoneme = per_calc(preds_ph, refs_ph)
    pred_sp = sum(sum(1 for t in p if t == SPACE_IDX) for p in preds)
    ref_sp = sum(sum(1 for t in r if t == SPACE_IDX) for r in refs)
    space_ratio = pred_sp / max(ref_sp, 1)
    tp = fp = fn = 0
    for p, r in zip(preds, refs):
        ps = sum(1 for t in p if t == SPACE_IDX)
        rs = sum(1 for t in r if t == SPACE_IDX)
        tp += min(ps, rs)
        fp += max(0, ps - rs)
        fn += max(0, rs - ps)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-6)
    return {'per_all': per_all, 'per_phoneme': per_phoneme,
            'space_ratio': space_ratio, 'space_f1': f1}

def cosine_warmup(ep, warm, total, min_r=0.01):
    if ep < warm:
        return max(0.01, ep / warm)
    p = (ep - warm) / max(1, total - warm)
    return max(min_r, 0.5 * (1 + math.cos(math.pi * p)))

def upload(ep):
    for f in glob.glob(f'{MDIR}/*.pt'):
        shutil.copy(f, f'{UDIR}/{os.path.basename(f)}')
    print(f'[upload ep{ep}] baslatildi -> {DATASET_ID}')
    r = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', UDIR,
         '-m', f'v1cc ep{ep}', '--dir-mode', 'zip'],
        capture_output=True, text=True, timeout=1800)
    if r.returncode == 0:
        print(f'[upload ep{ep}] VERSION OK')
    else:
        print(f'[upload ep{ep}] VERSION FAIL (rc={r.returncode})')
        stderr_lower = (r.stderr or '').lower()
        if 'not found' in stderr_lower or 'does not exist' in stderr_lower or '404' in stderr_lower:
            print(f'[upload ep{ep}] Dataset yok, CREATE deneniyor...')
            cr = subprocess.run(
                ['kaggle', 'datasets', 'create', '-p', UDIR, '--dir-mode', 'zip'],
                capture_output=True, text=True, timeout=1800)
            print(f'[upload ep{ep}] CREATE rc={cr.returncode}')
            if cr.stdout: print(f'  stdout: {cr.stdout[-300:]}')
            if cr.stderr: print(f'  stderr: {cr.stderr[-300:]}')

# Optimizer
fe_p = list(model.frontend3D.parameters()) + list(model.trunk.parameters())
be_p = [p for n, p in model.named_parameters() if not n.startswith(('frontend3D', 'trunk'))]
opt = AdamW([{'params': fe_p, 'lr': LR_FE}, {'params': be_p, 'lr': LR_BE}],
            betas=(0.9, 0.98), weight_decay=0.03)
sched = LambdaLR(opt, [lambda e: cosine_warmup(e, WARMUP, EPOCHS),
                       lambda e: cosine_warmup(e, WARMUP, EPOCHS)])
ctc_crit = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
scaler = torch.amp.GradScaler('cuda')

# v1cc: Label smoothing (Pereyra et al. 2017 — confidence penalty)
# loss_uniform = -mean(log_probs) → tensor, gradient akar
# loss_total = (1-alpha)*ctc + alpha*loss_uniform
print(f'Label smoothing: alpha={LABEL_SMOOTHING} (Pereyra confidence penalty)')

# ============ RESUME LOGIC ============
start_ep = 1
best_per = 1.0
best_f1 = 0.0

if not FORCE_SCRATCH:
    ckpts = ([CP] if os.path.exists(CP) else []) + glob.glob(
        '/kaggle/input/**/lippy-phoneme-v1cc-full-checkpoint/**/checkpoint.pt', recursive=True)
    ckpts = sorted(set(ckpts), key=lambda p: -os.path.getmtime(p) if os.path.exists(p) else 0)
    if ckpts:
        print(f'Resume candidates ({len(ckpts)}):')
        for p in ckpts[:3]:
            print(f'  - {p}')
        try:
            ck = torch.load(ckpts[0], map_location=DEVICE, weights_only=False)
            ckpt_vocab = ck['model_state_dict'].get('ctc_fc.weight', torch.zeros(1)).shape[0]
            cand_best_per = ck.get('best_per', 1.0)
            cand_epoch = ck.get('epoch', 0)
            if ckpt_vocab != NUM_PHONEMES:
                print(f'UYARI: Checkpoint vocab {ckpt_vocab} != {NUM_PHONEMES}, RESUME ATLANDI')
            elif cand_best_per > MAX_RESUME_PER:
                print(f'UYARI: Checkpoint patlamis (best_per={cand_best_per:.1%}), RESUME ATLANDI')
            else:
                model.load_state_dict(ck['model_state_dict'])
                start_ep = cand_epoch + 1
                best_per = cand_best_per
                best_f1 = ck.get('best_f1', 0.0)
                try:
                    opt.load_state_dict(ck['opt'])
                    sched.load_state_dict(ck['sched'])
                except Exception:
                    pass
                if 'scaler' in ck:
                    scaler.load_state_dict(ck['scaler'])
                print(f'Resume OK: epoch {cand_epoch} (best_per={best_per:.1%} best_f1={best_f1:.2f})')
        except Exception as e:
            print(f'Resume HATASI: {e} — sifirdan baslayacak')
    else:
        print('Onceki checkpoint yok — Auto-AVSR transfer ile sifirdan')
else:
    print('FORCE_SCRATCH=True — v1cc sifirdan basliyor (dim_ff degisti, eski ckpt uyumsuz)')

print(f'\nFE: {sum(p.numel() for p in fe_p):,} (lr={LR_FE})')
print(f'BE: {sum(p.numel() for p in be_p):,} (lr={LR_BE})')
print(f'Vocab: {NUM_PHONEMES} (SPACE DAHIL)')
print(f'DIM_FF: {DIM_FF} (v1c: 1024)')
print(f'Label smoothing: {LABEL_SMOOTHING} (confidence penalty)')
print(f'Speed perturbation: {SPEED_FACTORS}')
print(f'Mean/Std: {MEAN}/{STD}')
print(f'--- v1cc-FULL Egitim basliyor (start_ep={start_ep}) ---')
print()

epoch_history = []

for epoch in range(start_ep, EPOCHS + 1):
    AugState.current_epoch = epoch
    aug_active = epoch >= WARMUP_EPOCHS
    color_active = epoch >= COLOR_JITTER_FROM_EP and ENABLE_COLOR_JITTER
    entropy_alpha_eff = ALPHA_ENT if aug_active else 0.0

    if epoch < COLOR_JITTER_FROM_EP:
        mode_tag = 'WARM'
    elif epoch < WARMUP_EPOCHS:
        mode_tag = 'CLR '
    else:
        mode_tag = 'FULL'

    t0 = time.time()
    model.train()
    tl_s = tc = nan = 0
    opt.zero_grad(set_to_none=True)

    for step, (rois, tgts, il, tl) in enumerate(trl):
        rois, tgts = rois.to(DEVICE), tgts.to(DEVICE)
        try:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                lp = model(rois, il.to(DEVICE))

            log_probs = lp.float()  # (T, B, V)
            loss_ctc = ctc_crit(log_probs, tgts, il, tl)

            # v1cc: Label smoothing — Pereyra et al. 2017 confidence penalty
            # KL(uniform || model) = -mean(log_probs) — TENSOR, gradient akar!
            loss_uniform = -log_probs.mean()
            loss_total = (1.0 - LABEL_SMOOTHING) * loss_ctc + LABEL_SMOOTHING * loss_uniform

            if entropy_alpha_eff > 0:
                probs = log_probs.exp()
                entropy = -(probs * log_probs).sum(dim=-1).mean()
                loss = (loss_total - entropy_alpha_eff * entropy) / ACCUM
            else:
                loss = loss_total / ACCUM
        except RuntimeError as e:
            if 'out of memory' in str(e):
                torch.cuda.empty_cache(); nan += 1; continue
            raise
        if torch.isnan(loss) or torch.isinf(loss):
            nan += 1; continue
        scaler.scale(loss).backward()
        if (step + 1) % ACCUM == 0 or (step + 1) == len(trl):
            scaler.unscale_(opt)
            gn = nn.utils.clip_grad_norm_(model.parameters(), GCLIP)
            if torch.isfinite(gn):
                scaler.step(opt)
                tl_s += loss.item() * ACCUM
                tc += 1
            else:
                nan += 1
            scaler.update()
            opt.zero_grad(set_to_none=True)

    # Val
    model.eval()
    preds = []
    refs = []
    vl_s = vc = 0
    with torch.no_grad():
        for rois, tgts, il, tl in vll:
            rois = rois.to(DEVICE)
            lp = model(rois, il.to(DEVICE))
            vl = ctc_crit(lp.float(), tgts, il, tl)
            if not (torch.isnan(vl) or torch.isinf(vl)):
                vl_s += vl.item(); vc += 1
            for i in range(rois.shape[0]):
                preds.append(greedy_ctc(lp[:il[i], i]))
            off = 0
            for i in range(rois.shape[0]):
                refs.append(tgts[off:off + tl[i]].tolist())
                off += tl[i].item()

    train_loss = tl_s / max(tc, 1)
    val_loss = vl_s / max(vc, 1)
    m = v1c_metrics(preds, refs)

    if m['per_all'] < best_per:
        best_per = m['per_all']
        torch.save(model.state_dict(), BP)
        print(f'  [BEST] PER {best_per:.1%} kaydedildi')
    if m['space_f1'] > best_f1:
        best_f1 = m['space_f1']

    sched.step()
    ns = f' NaN:{nan}' if nan else ''
    gap = train_loss - val_loss
    print(f'[{mode_tag}] Ep {epoch:2d}/{EPOCHS} | TrL:{train_loss:.3f} | VaL:{val_loss:.3f} | gap:{gap:+.2f} | '
          f'PER-all:{m["per_all"]:.1%} ph:{m["per_phoneme"]:.1%} | SP F1:{m["space_f1"]:.2f} r:{m["space_ratio"]:.2f} | '
          f'{int(time.time() - t0)}s{ns}')

    if epoch % 5 == 0 or epoch == 1 or epoch == COLOR_JITTER_FROM_EP or epoch == WARMUP_EPOCHS:
        for i in range(min(2, len(preds))):
            print(f'  H: "{phonemes_to_text(refs[i])[:80]}"')
            print(f'  T: "{phonemes_to_text(preds[i])[:80]}"')

    epoch_history.append({'epoch': epoch, 'per_all': m['per_all'], 'per_phoneme': m['per_phoneme'],
                          'space_f1': m['space_f1'], 'val_loss': val_loss})

    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'scaler': scaler.state_dict(),
                'best_per': best_per, 'best_f1': best_f1}, CP)
    
    if epoch % UPLOAD_EVERY == 0:
        upload(epoch)

# Final upload
upload(EPOCHS)

# ============ FINAL OZET ============
print('\n' + '=' * 70)
print('v1cc-FULL TRAINING SONUC')
print('=' * 70)
print(f'\n{"Epoch":>6} | {"PER-all":>8} | {"PER-ph":>8} | {"SP F1":>6} | {"VaL":>6}')
print('-' * 50)
for h in epoch_history:
    print(f'{h["epoch"]:>6} | {h["per_all"]:>7.1%} | {h["per_phoneme"]:>7.1%} | {h["space_f1"]:>6.2f} | {h["val_loss"]:>6.3f}')
print('-' * 50)
print(f'{"BEST":>6} | {best_per:>7.1%} | {"-":>8} | {best_f1:>6.2f}')

print(f'\nKARSILASTIRMA:')
print(f'  v1c ep25 (baseline):      PER-all 49.1% (WER 78.7% beam+KenLM)')
print(f'  v1cc (speed+LS+dim2048):  PER-all {best_per:.1%}')
print(f'  Iyilestirme: {(0.491 - best_per)*100:+.1f} puan vs v1c')

print(f'\nv1cc YENILIKLER:')
print(f'  - dim_ff: 1024 -> 2048 (2x temporal kapasite)')
print(f'  - Speed perturbation: {SPEED_FACTORS} (efektif 3x veri)')
print(f'  - Label smoothing: {LABEL_SMOOTHING} (Pereyra confidence penalty — TENSOR)')
print(f'  - Mean/Std: {MEAN}/{STD} (dataset gercek istatistigi)')

print(f'\nSONRAKI ADIM:')
print(f'  1. lippy-vallr-decode-v1cc.ipynb: KenLM ile WER hesapla')
print(f'  2. Beklenen: PER 41-45%, WER 70-74% (beam+KenLM)')
print('=' * 70)